In [1]:
from pydantic import BaseModel
from typing import Literal

In [2]:
class SupportTicket(BaseModel):
    category: Literal["BILLING", "TECHNICAL", "OTHER"]
    priority: Literal["LOW", "MEDIUM", "HIGH"]
    amount: float

In [3]:
ticket1 = SupportTicket(
    category="BILLING",
    priority="HIGH",
    amount=2000
)

print(ticket1)
print(type(ticket1.amount))

category='BILLING' priority='HIGH' amount=2000.0
<class 'float'>


In [4]:
ticket2 = SupportTicket(
    category="BILLING",
    priority="HIGH",
    amount="3000"
)

print(ticket2)
print(type(ticket2.amount))

category='BILLING' priority='HIGH' amount=3000.0
<class 'float'>


In [5]:
ticket3 = SupportTicket(
    category="BILLING",
    priority="URGENT",
    amount="3000"
)

print(ticket3)
print(type(ticket3.amount))

ValidationError: 1 validation error for SupportTicket
priority
  Input should be 'LOW', 'MEDIUM' or 'HIGH' [type=literal_error, input_value='URGENT', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error

In [7]:
import json

response_a = '''
{
    "category": "BILLING",
    "priority": "HIGH",
    "amount": 2000
}
'''

parsed = json.loads(response_a)
print(parsed)

{'category': 'BILLING', 'priority': 'HIGH', 'amount': 2000}


In [8]:
import json
from pydantic import ValidationError

def validate_ticket(llm_response: str):
    try:
        parsed = json.loads(llm_response)
        ticket = SupportTicket.model_validate(parsed)
        return ticket

    except json.JSONDecodeError as e:
        print(f"JSON parsing failed: {e}")
        return

    except ValidationError as e:
        print(f"Schema validation failed: {e}")
        return

In [9]:
# Test 1: Valid JSON and valid schema
test_1 = '''
{
    "category": "BILLING",
    "priority": "HIGH",
    "amount": 2000
}
'''

# Test 2: Invalid JSON (trailing comma)
test_2 = '''
{
    "category": "TECHNICAL",
    "priority": "MEDIUM",
    "amount": 1500,
}
'''

# Test 3: Valid JSON but invalid schema
test_3 = '''
{
    "category": "BILLING",
    "priority": "URGENT",
    "amount": 3000
}
'''

In [10]:
print("Test 1:", validate_ticket(test_1))
print("Test 2:", validate_ticket(test_2))
print("Test 3:", validate_ticket(test_3))

Test 1: category='BILLING' priority='HIGH' amount=2000.0
JSON parsing failed: Expecting property name enclosed in double quotes: line 6 column 1 (char 78)
Test 2: None
Schema validation failed: 1 validation error for SupportTicket
priority
  Input should be 'LOW', 'MEDIUM' or 'HIGH' [type=literal_error, input_value='URGENT', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
Test 3: None
